# Reform polling bias

For every electoral event in our snapshot (by-elections + curated local-election PNS), compare the pre-event 7-day national poll mean for Reform against the actual Reform result. Aggregate to a single recommended `--reform-polling-correction-pp` value.

**Caveats:** by-elections and local elections are weighted equally (see spec §5.2). Per-pollster bias is shown but is descriptive — most pollsters appear in too few events for statistically powerful estimates.

In [ ]:
import os
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repo root (no pyproject.toml in cwd or any parent)")

os.chdir(_find_repo_root())

def _latest_snapshot_hash() -> str:
    """Return the content hash of the most recent snapshot. Filenames are
    YYYY-MM-DD__v<schema>__<hash>.sqlite, so lexical sort = chronological."""
    snaps = sorted(Path("data/snapshots").glob("*.sqlite"))
    if not snaps:
        raise FileNotFoundError("no snapshots in data/snapshots/; run /snapshot first")
    return snaps[-1].stem.split("__")[-1]

def _pick_prediction(strategy_marker: str, label: str) -> Path:
    """Select the prediction file for the LATEST snapshot whose name contains
    strategy_marker AND label. Prediction filenames are
    <snap_hash>__<strategy>__<config_hash>__<label>.sqlite. Fails loud if 0 or
    >1 files match — the previous sorted(glob)[-1] silently picked an
    alphabetically-last file, which became nondeterministic once sweep_* runs
    or older snapshots' predictions shared the directory."""
    snap_hash = _latest_snapshot_hash()
    pred_dir = Path("data/predictions")
    matches = [
        p for p in pred_dir.glob(f"{snap_hash}__*{strategy_marker}*.sqlite")
        if label in p.name
    ]
    if not matches:
        raise FileNotFoundError(
            f"no prediction in {pred_dir} for snapshot {snap_hash} matching "
            f"strategy={strategy_marker!r} label={label!r}; run /predict first"
        )
    if len(matches) > 1:
        names = sorted(p.name for p in matches)
        raise RuntimeError(
            f"multiple predictions for snapshot {snap_hash} match "
            f"strategy={strategy_marker!r} label={label!r}: {names}; "
            f"remove duplicates or pass a more specific label"
        )
    return matches[0]

In [ ]:
from pathlib import Path
from data_engine.sources.local_elections import load_local_elections
from prediction_engine.snapshot_loader import Snapshot
from prediction_engine.analysis.poll_bias import compute_reform_bias, write_bias_json

snap_path = sorted(Path("data/snapshots").glob("*.sqlite"))[-1]
snap = Snapshot(snap_path)
local_yaml = Path("data/hand_curated/local_elections.yaml")
local_events = load_local_elections(local_yaml)
print(f"Snapshot: {snap_path.name}")
print(f"Local-election events loaded: {len(local_events)}")
print(f"By-election events in snapshot: {len(snap.byelections_events)}")

In [ ]:
import pandas as pd
result = compute_reform_bias(snap, local_elections=local_events)
per_event_df = pd.DataFrame(result.per_event)
per_event_df

In [ ]:
per_pollster_df = pd.DataFrame.from_dict(result.per_pollster, orient="index")
per_pollster_df.sort_values("mean_bias_pp", ascending=False)

In [ ]:
print(f"Aggregate Reform polling bias: {result.aggregate_bias_pp:+.2f} pp")
print(f"Events used: {result.n_events_used} (with polls in window: {result.n_events_with_polls})")
print()
print(f"Recommended CLI flag:")
print(f"  --reform-polling-correction-pp {result.recommended_reform_polling_correction_pp:+.2f}")

In [ ]:
out_path = Path("data/derived/reform_polling_bias.json")
write_bias_json(result, snap, local_elections_yaml_path=local_yaml, out_path=out_path)
print(f"Wrote {out_path}")

A positive aggregate means pollsters under-state Reform; pass `+aggregate` to `seatpredict-predict --reform-polling-correction-pp`. Negative means pollsters over-state — pass the negative value. Per-pollster numbers below `n_events_with_polls = 3` are flagged `low` reliability and should be read as descriptive only.